In [ ]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import classification_report, accuracy_score

In [ ]:
df = pd.read_csv('Churn_Modelling.csv')
df.sample(2)

In [ ]:
# Check class balance
df['Exited'].value_counts()

In [ ]:
# Drop columns that are not useful for prediction
df = df.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)

X = df.drop('Exited', axis=1)
y = df['Exited']

x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# Preprocessing: encode categorical columns, scale numerical columns
trf = ColumnTransformer(transformers=[
    ('cat',  OrdinalEncoder(), ['Geography', 'Gender']),
    ('num',  StandardScaler(), ['CreditScore', 'Age', 'Tenure', 'Balance',
                                'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary'])
], remainder='passthrough')

## RandomForest — GridSearchCV

GridSearchCV tries every combination of hyperparameters we give it and picks the best one using cross-validation.

| Parameter | What it controls |
|---|---|
| `n_estimators` | Number of trees in the forest |
| `max_depth` | How deep each tree can grow |
| `min_samples_split` | Minimum samples needed to split a node |

In [ ]:
# Build the pipeline
rf_pipe = Pipeline(steps=[
    ('trf',   trf),
    ('model', RandomForestClassifier(random_state=42))
])

# Define the hyperparameter grid
# NOTE: prefix 'model__' tells GridSearchCV these params belong to the 'model' step
rf_params = {
    'model__n_estimators':    [100, 200],
    'model__max_depth':       [5, 7, 10],
    'model__min_samples_split': [2, 5]
}

# cv=5 means 5-fold cross-validation
# scoring='accuracy' is what we optimise for
# n_jobs=-1 uses all CPU cores to speed things up
rf_grid = GridSearchCV(rf_pipe, rf_params, cv=5, scoring='accuracy', n_jobs=-1, verbose=1)
rf_grid.fit(x_train, y_train)

print('\nBest Parameters:', rf_grid.best_params_)
print('Best CV Accuracy:', round(rf_grid.best_score_, 4))

In [ ]:
# Evaluate on the test set using the best model found
y_pred_rf = rf_grid.predict(x_test)

print('RandomForest Test Accuracy:', accuracy_score(y_test, y_pred_rf))
print()
print(classification_report(y_test, y_pred_rf))

## Decision Tree — GridSearchCV

| Parameter | What it controls |
|---|---|
| `max_depth` | How deep the tree can grow |
| `min_samples_split` | Minimum samples needed to split a node |
| `criterion` | How to measure split quality (gini or entropy) |

In [ ]:
# Build the pipeline
dt_pipe = Pipeline(steps=[
    ('trf',   trf),
    ('model', DecisionTreeClassifier(random_state=42))
])

# Define the hyperparameter grid
dt_params = {
    'model__max_depth':        [5, 7, 10, None],
    'model__min_samples_split': [2, 5, 10],
    'model__criterion':        ['gini', 'entropy']
}

dt_grid = GridSearchCV(dt_pipe, dt_params, cv=5, scoring='accuracy', n_jobs=-1, verbose=1)
dt_grid.fit(x_train, y_train)

print('\nBest Parameters:', dt_grid.best_params_)
print('Best CV Accuracy:', round(dt_grid.best_score_, 4))

In [ ]:
# Evaluate on the test set
y_pred_dt = dt_grid.predict(x_test)

print('Decision Tree Test Accuracy:', accuracy_score(y_test, y_pred_dt))
print()
print(classification_report(y_test, y_pred_dt))

## Final Comparison

In [ ]:
print('='*40)
print('         MODEL COMPARISON')
print('='*40)
print(f'Random Forest  : {accuracy_score(y_test, y_pred_rf):.4f}')
print(f'Decision Tree  : {accuracy_score(y_test, y_pred_dt):.4f}')
print('='*40)

print('\nBest RF params :', rf_grid.best_params_)
print('Best DT params :', dt_grid.best_params_)